# TriNetra-AMRF — Fusion+TRC Training on M3FD (Colab)

Full training run of the TRC-gated fusion detector on the complete M3FD
train split, under the reference "Experimental Setting" table:

| Parameter | Value |
|---|---|
| Epochs | 150 (with early stopping, patience 15) |
| Batch size | 16 |
| Input size | 640x640 |
| Optimizer | SGD (lr0=0.01, momentum=0.937, weight_decay=0.0005) |
| LR schedule | Cosine decay, lr0=0.01 -> lr_final=1e-4 (lrf=0.01) |
| Workers | 8 |
| Augmentation | HSV-equivalent jitter + horizontal flip (mosaic not implemented — see note below) |
| Precision | FP32 (AMP off) |

**On mosaic:** the table calls for mosaic augmentation; this project does not
implement it. Mosaic would need real 4-image composition + bounding-box
remapping across *two* modalities (visible + thermal) simultaneously, and the
project's own Phase-5 design doc flags it as complicating the TRC-gating story
(four composited frames -> four different TRC scores to reconcile into one).
Color jitter (HSV-equivalent) and horizontal flip are enabled instead.

**Before running:** the M3FD zip should be on Google Drive with link-sharing
on ("Anyone with the link"). Runtime -> Change runtime type -> GPU (T4).

**If your runtime was ever restarted/reconnected:** re-run cells 1–6 from
the top before touching Section 7 — a fresh runtime has no repo, no
dataset, and no installed packages, even if a previous session's cell
outputs are still shown on screen.

## 1. Confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clone the repo

In [ ]:
!git clone https://github.com/SadiqueK78/Tri-Netra.git
%cd Tri-Netra

## 3. Install dependencies

`albumentations` pinned to 1.4.6 for the same reason as before (a newer
version pulls in `albucore`->`stringzilla`, which needs a C compiler to
build from source on platforms with no prebuilt wheel).

In [ ]:
!pip install -q "albumentations==1.4.6" --no-deps
!pip install -q ultralytics deep-sort-realtime pycocotools python-dotenv GPUtil nvidia-ml-py gdown

## 4. Download the M3FD dataset from Drive

By file ID via `gdown` — works regardless of which Google account owns the
zip or which account is signed into this Colab session, as long as the file
is shared as "Anyone with the link".

In [ ]:
# The file ID is the part between "/d/" and "/view" in the share URL.
M3FD_FILE_ID = "1ZIhoIpoz3CXVJxCsLJIgyQiM_NrDXZ6l"

!gdown --id $M3FD_FILE_ID -O M3FD.zip
!mkdir -p datasets/M3FD
!unzip -q M3FD.zip -d datasets/M3FD
!if [ -d datasets/M3FD/M3FD ]; then mv datasets/M3FD/M3FD/* datasets/M3FD/ && rmdir datasets/M3FD/M3FD; fi
!echo '--- expect Vis/ Ir/ Annotation/ below ---' && ls datasets/M3FD

## 5. Convert annotations and build the split

In [ ]:
!python datasets/convert_annotations.py --source m3fd --validate

In [ ]:
!python datasets/split_dataset.py --config configs/default.yaml

## 6. Pre-flight checks

Cheap sanity checks before spending real GPU time on 150 epochs.

In [ ]:
!python -m training.loss_adapter --self-test

In [ ]:
!python utils/check_fusion.py --test loss --split val

In [ ]:
# Quick 4-epoch overfit sanity check — confirms the SGD/640x640/AMP-off
# loop runs cleanly on THIS runtime before committing to the full 150 epochs.
!python training/train_fusion.py --overfit 16 --epochs 4

## 7. Train: Fusion+TRC, full M3FD train split, 150 epochs

This is the real run — the full 2,940-image M3FD train split (not a
subset), config-driven from `configs/default.yaml` (SGD, 640x640, AMP off,
150 epochs with early stopping at patience 15). Runs in the background
with output streamed to a log file so the cell doesn't block the notebook
and you can keep watching progress in the next cell.

The launch cell below sets `cwd` explicitly on the subprocess, so it works
correctly even if this cell is re-run on its own after a runtime restart
(as long as Sections 2–4 have been (re-)run at least once in the current
runtime, so the repo and dataset actually exist on disk).

In [ ]:
import subprocess, os

REPO_DIR = "/content/Tri-Netra"
assert os.path.isfile(os.path.join(REPO_DIR, "training", "train_fusion.py")), (
    f"{REPO_DIR}/training/train_fusion.py not found — re-run Section 2 "
    "(clone the repo) in this runtime first."
)

log_path = os.path.join(REPO_DIR, "fusion_trc_train.log")
proc = subprocess.Popen(
    ["python", "-u", "training/train_fusion.py", "--stage", "1", "--epochs", "150"],
    cwd=REPO_DIR,
    stdout=open(log_path, "w"), stderr=subprocess.STDOUT,
)
print(f"Training started, PID={proc.pid}. Log: {log_path}")
print("Run the next cell any time to see live progress (re-run it to refresh).")

In [ ]:
# Re-run this cell any time to see the latest progress.
!tail -n 40 /content/Tri-Netra/fusion_trc_train.log

In [ ]:
# Blocks until training finishes (or the runtime is interrupted). Optional —
# skip this and just keep re-running the tail cell above if you'd rather
# check in periodically instead of blocking the notebook.
proc.wait()
print("Training process exited with code", proc.returncode)

## 8. Show the CSV log and training curves

In [ ]:
import pandas as pd
log_df = pd.read_csv("/content/Tri-Netra/weights/fusion/training_log.csv")
log_df

In [ ]:
from IPython.display import Image as IPImage, display
display(IPImage("/content/Tri-Netra/weights/fusion/training_curves.png"))

## 9. Evaluate on the M3FD test split

In [ ]:
!python training/evaluate.py --checkpoint weights/fusion/best.pt --split test

## 10. Demo: run inference on one pair and show the detection image

In [ ]:
import glob
sample_vis = sorted(glob.glob('datasets/visible/test/*'))[0]
sample_name = sample_vis.split('/')[-1]
sample_thr = f'datasets/thermal/test/{sample_name}'

!python inference/predict_image.py \
    --checkpoint weights/fusion/best.pt \
    --visible "$sample_vis" --thermal "$sample_thr" \
    --output runs/predictions/colab_demo.jpg

In [ ]:
from IPython.display import Image as IPImage, display
display(IPImage('runs/predictions/colab_demo.jpg'))

## 11. Save everything to Drive

Colab's local disk is wiped when the runtime disconnects — copy the
checkpoint, CSV log, plot, and demo image to Drive before closing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUT_DIR = "/content/drive/MyDrive/TriNetra_FusionTRC_results"
os.makedirs(OUT_DIR, exist_ok=True)
!cp -r weights/fusion "$OUT_DIR/"
!cp runs/predictions/colab_demo.jpg "$OUT_DIR/"
print('Saved to', OUT_DIR)